In [1]:
import os, re, random
from collections import defaultdict, Counter
import numpy as np
import torch
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
class ADNIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = {'AD': 0, 'NC': 1}
        self.classes = list(self.class_to_idx.keys())
        self.samples = []

        for cls in self.classes:
            class_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(class_dir):
                continue
            label = self.class_to_idx[cls]
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    pid = self._extract_patient_id(fname)
                    self.samples.append((os.path.join(class_dir, fname), label, pid))

    def _extract_patient_id(self, filename):
        m = re.match(r"(\d{6})", filename)
        return m.group(1) if m else None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, _ = self.samples[idx]
        image = Image.open(path).convert("L")
        if self.transform:
            image = self.transform(image)
        return image, label

In [3]:
def patient_level_split(dataset, test_size=0.2, random_state=42):
    patient_to_indices = defaultdict(list)
    for idx, (_, label, pid) in enumerate(dataset.samples):
        if pid:
            patient_to_indices[pid].append(idx)
    unique_pids = list(patient_to_indices.keys())
    labels = [dataset.samples[patient_to_indices[pid][0]][1] for pid in unique_pids]

    train_pids, val_pids = train_test_split(unique_pids, test_size=test_size,
                                            random_state=random_state, stratify=labels)
    train_idx = [i for pid in train_pids for i in patient_to_indices[pid]]
    val_idx = [i for pid in val_pids for i in patient_to_indices[pid]]
    return train_idx, val_idx

In [5]:
def get_adni_transforms(mean=0.263, std=0.271):
    train_tf = transforms.Compose([
        transforms.Pad((8, 0, 8, 0)),
        transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.01, 0.05)),
        transforms.Normalize(mean=[mean], std=[std]),
    ])
    eval_tf = transforms.Compose([
        transforms.Pad((8, 0, 8, 0)),
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[mean], std=[std]),
    ])
    return train_tf, eval_tf

In [6]:
def create_datasets(train_dir, test_dir, val_split=0.2):
    train_tf, eval_tf = get_adni_transforms()
    base_dataset = ADNIDataset(train_dir)
    train_idx, val_idx = patient_level_split(base_dataset, val_split)
    train_ds = Subset(ADNIDataset(train_dir, train_tf), train_idx)
    val_ds   = Subset(ADNIDataset(train_dir, eval_tf), val_idx)
    test_ds  = ADNIDataset(test_dir, eval_tf)
    return train_ds, val_ds, test_ds

In [7]:
def create_dataloaders(train_ds, val_ds, test_ds, batch_size=128, num_workers=8):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader

In [8]:
def count_classes(dataset, name="Dataset"):
    """
    Count the number of samples per class in a dataset (works for ImageFolder or Subset).
    """
    if isinstance(dataset, torch.utils.data.Subset):
        # Access underlying dataset and the indices
        targets = np.array([dataset.dataset.samples[i][1] for i in dataset.indices])
    else:
        # Direct ImageFolder dataset
        targets = np.array([s[1] for s in dataset.samples])

    class_weights = []

    counts = Counter(targets)
    print(f"\n{name} class distribution:")
    for cls_idx, count in counts.items():
        class_name = dataset.dataset.classes[cls_idx] if isinstance(dataset, torch.utils.data.Subset) else dataset.classes[cls_idx]
        print(f"  {class_name}: {count}")

        class_weights.append(len(targets)/(len(counts.items())*count))

    print(f"  Total: {len(targets)}")

    return class_weights

In [9]:
def show_samples(dataset, num_samples=10, title="Sample Images"):
    """Display sample images from a dataset"""
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle(title, fontsize=16)

    for i in range(num_samples):
        image, label = dataset[random.randint(0, len(dataset)-1)]
        row, col = divmod(i, 5)
        image = image * 0.271 + 0.263  # denormalize
        image = image.squeeze(0)
        axes[row, col].imshow(image, cmap='gray')
        axes[row, col].set_title(f'Label: {label}')
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()